# Project 1: Building My LLM Playground

In this project, I built a simple Large Language Model (LLM) playground to better understand how LLMs work internally. Instead of using high-level frameworks right away, I wanted to explore the core workflow of an LLM, from receiving a text prompt to generating a meaningful response.

The primary objective of this project is to gain a practical understanding of the fundamental concepts behind LLMs. By working directly with the underlying components, I can observe how the model processes input, predicts tokens, and produces output. This hands-on approach helps build a strong conceptual foundation before moving on to frameworks that abstract much of this functionality.

In future projects, I'll work with tools like **Ollama** and **LangChain**, which simplify model interaction and application development. However, understanding the fundamentals first makes it much easier to appreciate what these frameworks are doing behind the scenes.

---

## Environment Setup

For this project, I used **Google Colab** as my development environment. It provides a cloud-based Jupyter Notebook interface where I can write and execute Python code without installing any software locally. After creating a new notebook in Google Colab, I uploaded this project notebook and used it to perform all the experiments and implementations throughout the project.


If I want to run this project on my local machine instead of Google Colab, I first need to set up a dedicated Python environment. Creating a separate environment helps keep the project dependencies organized and avoids conflicts with other Python projects.

I opened a terminal in the project directory and followed one of the methods below to install the required packages.

### Step 1: Set Up the Project Environment

#### Option 1: Using Conda

Conda is a package and environment manager that makes it easy to create isolated environments and install all the required dependencies.

```bash
# Create the project environment and activate it
conda env create -f environment.yaml && conda activate llm_playground
```

#### Option 2: Using uv

I also explored `uv`, which is a fast Python package and virtual environment manager. It provides a lightweight and efficient way to create environments and install dependencies.

```bash
# Install uv (skip this step if it is already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create a virtual environment and install the required packages
uv venv .venv && source .venv/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register the Environment as a Jupyter Kernel

If the newly created environment does not appear in Jupyter Notebook or JupyterLab, I can register it manually using the following command.

```bash
python -m ipykernel install --user --name=llm_playground --display-name "llm_playground"
```

After registering the environment, I selected the **llm_playground** kernel from the Jupyter kernel menu and continued working with the notebook.


## Learning Objectives

Before starting the implementation, I wanted to define the concepts I planned to explore in this notebook. These objectives helped me stay focused while experimenting with different parts of a Large Language Model and understanding how each component contributes to the overall text generation process.

* Learn how text is converted into tokens before it is processed by the model.
* Load a pretrained GPT-2 model and examine how it can be used for text generation.
* Understand how logits are transformed into probabilities for predicting the next token.
* Explore the number of parameters in the model to get an idea of its overall size and complexity.
* Compare greedy decoding with top-p (nucleus) sampling and observe how different decoding strategies influence the generated output.
* Understand the differences between GPT-2 and modern instruction-following language models.
* Gain a basic understanding of inference engines and their role in serving language models efficiently.


In [1]:
# Import the libraries required for this project and verify that they are available.
import torch, transformers, tiktoken

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("All required libraries are available. The environment is ready for the next steps.")

torch 2.11.0+cu128 | transformers 5.12.0
All required libraries are available. The environment is ready for the next steps.


# 1: Tokenization

Language models cannot understand plain text directly. Before processing any input, the text must be converted into numerical token IDs. This conversion process is called **tokenization**, and it is one of the fundamental steps in how an LLM processes text.

There are three commonly used tokenization approaches:

1. Word-level
2. Character-level
3. Subword-level

### 1.1 - Word-level tokenization

I started by exploring word-level tokenization, where a sentence is split into individual words based on whitespace. Each unique word is then assigned a token ID. In the next step, I implemented a simple word-level tokenizer by creating a vocabulary along with `encode` and `decode` functions.


In [3]:
# I created a small sample corpus to understand how a vocabulary is generated.
# In real applications, language models are trained on much larger text collections.

corpus = [
    "learning language models is exciting",
    "python makes machine learning easier",
    "artificial intelligence is transforming many industries",
    "every token represents a piece of text",
    "deep learning models require quality data",
    "natural language processing helps computers understand text",
    "building projects improves programming skills",
    "experimentation is the best way to learn new concepts",
    "open source libraries simplify machine learning development",
    "practice and consistency lead to better results"
]

# Initialize the vocabulary and the mappings between words and token IDs.
vocab = []
word2id = {}
id2word = {}


PAD, UNK = "[PAD]", "[UNK]"

# Collect all unique words from the corpus.
words = set()
for doc in corpus:
    words.update(doc.lower().split())

# Create the vocabulary by adding special tokens first.
vocab = [PAD, UNK] + sorted(words)

# Assign a unique ID to every word.
word2id = {w: i for i, w in enumerate(vocab)}

# Create a reverse mapping from IDs back to words.
id2word = {i: w for w, i in word2id.items()}

print(f"Vocabulary contains {len(vocab)} unique tokens.")
print("Generated vocabulary:", vocab)

Vocabulary contains 56 unique tokens.
Generated vocabulary: ['[PAD]', '[UNK]', 'a', 'and', 'artificial', 'best', 'better', 'building', 'computers', 'concepts', 'consistency', 'data', 'deep', 'development', 'easier', 'every', 'exciting', 'experimentation', 'helps', 'improves', 'industries', 'intelligence', 'is', 'language', 'lead', 'learn', 'learning', 'libraries', 'machine', 'makes', 'many', 'models', 'natural', 'new', 'of', 'open', 'piece', 'practice', 'processing', 'programming', 'projects', 'python', 'quality', 'represents', 'require', 'results', 'simplify', 'skills', 'source', 'text', 'the', 'to', 'token', 'transforming', 'understand', 'way']


In [4]:
# Step 2: Define encode and decode functions

def encode(text):
    # Convert the input text into a sequence of token IDs.
    return [word2id.get(w, word2id[UNK]) for w in text.lower().split()]


def decode(ids):
    # Convert token IDs back into readable text.
    return " ".join(id2word[i] for i in ids if i != word2id[PAD])

In [5]:
# Step 3: Test the tokenizer using a sample sentence.
# This example also includes words that are not present in the vocabulary.

sample = "it works only on text"

ids = encode(sample)
recovered = decode(ids)

print("\nInput Text :", sample)
print("Token IDs  :", ids)
print("Decoded Text:", recovered)


Input Text : it works only on text
Token IDs  : [1, 1, 1, 1, 49]
Decoded Text: [UNK] [UNK] [UNK] [UNK] text


Although word-level tokenization is easy to implement, I observed a couple of important limitations:

1. **Large vocabulary:** Every unique word needs its own token. As the dataset grows, the vocabulary becomes much larger, increasing the memory required by the model.

2. **Unknown words:** If the model encounters a word that is not present in the vocabulary, it cannot assign a valid token ID. Such words are typically represented using a special `[UNK]` token.

To address these limitations, I next explored character-level tokenization, where each individual character is treated as a token instead of an entire word.


### 1.2 - Character-level tokenization

Instead of treating an entire word as a token, this approach assigns a unique token ID to every individual character. This includes letters, spaces, punctuation marks, and other symbols.

To understand how this works, I rebuilt the tokenizer using the same corpus from the previous example. For this implementation, I limited the character set to English alphabets and common punctuation symbols to keep the tokenizer simple.


In [6]:
import string

# Build a character-level vocabulary using letters, digits, spaces, and punctuation.
vocab = []
char2id = {}
id2char = {}

letters = list(string.ascii_lowercase + string.ascii_uppercase)
digits = list(string.digits)
symbols = [" "] + list(string.punctuation)
special = ["[PAD]", "[UNK]"]

vocab = special + letters + digits + symbols

# Create mappings between characters and their IDs.
char2id = {ch: idx for idx, ch in enumerate(vocab)}
id2char = {idx: ch for ch, idx in char2id.items()}

print(f"Vocabulary size: {len(vocab)}")
print("Character vocabulary created successfully.")

Vocabulary size: 97
Character vocabulary created successfully.


In [7]:
# Step 2: Implement encode() and decode() functions to convert between text and IDs.

def encode(text):
    # Convert each character in the input text into its corresponding ID.
    unk_id = char2id["[UNK]"]
    return [char2id.get(ch, unk_id) for ch in text]


def decode(ids):
    # Reconstruct the original text from the character IDs.
    return "".join(id2char[i] for i in ids if i != char2id["[PAD]"])

In [8]:
# Step 3: Test the character-level tokenizer with a sample input.

sample = "Hello, AI 2026!"

ids = encode(sample)
recovered = decode(ids)

print("\nInput Text :", sample)
print("Token IDs  :", ids)
print("Decoded Text:", recovered)


Input Text : Hello, AI 2026!
Token IDs  : [35, 6, 13, 13, 16, 76, 64, 28, 36, 64, 56, 54, 56, 60, 65]
Decoded Text: Hello, AI 2026!


Character-level tokenization addresses the issue of unknown words, but it also has a few drawbacks:

1. **Longer input sequences:** Each word is split into multiple characters, resulting in a larger number of tokens for the same text.
2. **Limited meaning per token:** Individual characters carry very little information, so the model must combine many characters to understand a word.
3. **Higher processing cost:** Longer token sequences require more computation during both training and inference.

To overcome these limitations, the next approach uses **subword-level tokenization**, which provides a better balance between vocabulary size and sequence length.


### 1.3 - Subword-level tokenization

After exploring word-level and character-level approaches, I moved on to subword-level tokenization. Methods such as `Byte-Pair Encoding (BPE)`, `WordPiece`, and `SentencePiece` split words into smaller meaningful units. For example, the word **unbelievable** can be represented as **["un", "believ", "able"]**. This approach helps reduce vocabulary size while still preserving meaningful text representations.

The basic idea behind the BPE algorithm is:

1. Start with individual characters or bytes as the initial tokens.
2. Count the frequency of adjacent token pairs in the training data.
3. Merge the most frequently occurring pair into a new token.

These steps are repeated until the vocabulary reaches the required size.

Instead of implementing BPE from scratch, I used a pretrained tokenizer from `GPT-2`. Since it has already learned an effective vocabulary from a large text corpus, it provides a practical way to observe how subword tokenization works in real language models.


In [10]:
from transformers import AutoTokenizer

# Load the pretrained GPT-2 tokenizer from Hugging Face.
# This tokenizer uses Byte-Pair Encoding (BPE) for tokenization.
# Reference: https://huggingface.co/docs/transformers/en/model_doc/gpt2

bpe_tok = AutoTokenizer.from_pretrained("gpt2")

print("Vocabulary Size:", bpe_tok.vocab_size)
print("Special Tokens:", bpe_tok.all_special_tokens)

Vocabulary Size: 50257
Special Tokens: ['<|endoftext|>']


In [12]:
# Step 2: Encode a sample sentence and observe how BPE splits it into subword tokens.
# Reference: https://huggingface.co/docs/transformers/en/model_doc/gpt2

sample = "Tokenization improves language model efficiency."

ids = bpe_tok.encode(sample)
recovered = bpe_tok.decode(ids)

print("\nInput Text :", sample)
print("Token IDs  :", ids)
print("Subword Tokens:", bpe_tok.convert_ids_to_tokens(ids))
print("Decoded Text :", recovered)


Input Text : Tokenization improves language model efficiency.
Token IDs  : [30642, 1634, 19575, 3303, 2746, 9332, 13]
Subword Tokens: ['Token', 'ization', 'Ġimproves', 'Ġlanguage', 'Ġmodel', 'Ġefficiency', '.']
Decoded Text : Tokenization improves language model efficiency.


### 1.4 - TikToken

After exploring basic tokenization techniques, I wanted to understand how production-grade tokenizers work. `tiktoken` is the tokenizer library developed by OpenAI and is optimized for speed and efficient token counting in GPT models.

In this section, I compare the tokenizers used by different GPT model families. This helps illustrate how tokenization has evolved to efficiently handle a wider range of text, including Unicode characters, emojis, and special symbols, while maintaining good performance.

In the next cell, I use `tiktoken` to load different encodings and compare how each tokenizer processes the same input text.

**Reference:** https://github.com/openai/tiktoken


In [13]:
import tiktoken

# Compare the tokenization behavior of GPT-2 and GPT-4 style encodings.

# Step 1: Load the tokenizers.
# Reference: https://github.com/openai/tiktoken

encodings = [
    ("gpt2", tiktoken.get_encoding("gpt2")),
    ("cl100k_base", tiktoken.get_encoding("cl100k_base")),
]

# Step 2: Encode the same sentence using both tokenizers and compare the results.
sentence = "The AI assistant summarized the report in seconds."

for name, enc in encodings:
    print(f"\n=== {name} ===")
    print("Vocabulary Size:", enc.n_vocab)

    # Encode the sample sentence.
    ids = enc.encode(sentence)
    tokens = [enc.decode([i]) for i in ids]

    print(f"Number of Tokens: {len(ids)}")
    print(list(zip(tokens, ids)))

    # Display a few sample token IDs from the vocabulary.
    sample_ids = [0, 1, 2, 198, 50256]
    print("Sample Vocabulary Tokens:")
    print([(enc.decode([i]), i) for i in sample_ids])

# Step 3: Display the special tokens supported by each tokenizer.

print("\n#### Special Tokens ####")

for name, enc in encodings:
    print(f"\n=== {name} ===")
    special_set = getattr(enc, "special_tokens_set", None)

    if special_set is not None:
        print("Special Tokens:", sorted(special_set))
    else:
        print("Special Tokens: Not available for this encoding.")


=== gpt2 ===
Vocabulary Size: 50257
Number of Tokens: 9
[('The', 464), (' AI', 9552), (' assistant', 8796), (' summarized', 31880), (' the', 262), (' report', 989), (' in', 287), (' seconds', 4201), ('.', 13)]
Sample Vocabulary Tokens:
[('!', 0), ('"', 1), ('#', 2), ('\n', 198), ('<|endoftext|>', 50256)]

=== cl100k_base ===
Vocabulary Size: 100277
Number of Tokens: 9
[('The', 791), (' AI', 15592), (' assistant', 18328), (' summarized', 69729), (' the', 279), (' report', 1934), (' in', 304), (' seconds', 6622), ('.', 13)]
Sample Vocabulary Tokens:
[('!', 0), ('"', 1), ('#', 2), ('\n', 198), ('parable', 50256)]

#### Special Tokens ####

=== gpt2 ===
Special Tokens: ['<|endoftext|>']

=== cl100k_base ===
Special Tokens: ['<|endofprompt|>', '<|endoftext|>', '<|fim_middle|>', '<|fim_prefix|>', '<|fim_suffix|>']


Feel free to modify the sample input and observe how the tokenizer output changes. Some interesting cases to explore include:

* Sentences with emojis, punctuation, or special symbols.
* Code snippets, JSON, or other structured text.
* Text written in different languages such as Japanese, French, or Arabic.

As an additional exercise, you can also try implementing the BPE algorithm on a small corpus to understand how subword tokens are created by repeatedly merging the most frequent token pairs.

### 1.5 - Key Takeaways

* **Word-level tokenization:** Easy to understand and implement, but requires a large vocabulary and struggles with unseen words.
* **Character-level tokenization:** Can represent any text, but generates longer token sequences that require more computation.
* **Subword (BPE) tokenization:** Provides a good balance between vocabulary size and sequence length, making it the preferred choice for modern language models.
* **TikToken:** An efficient tokenizer used by OpenAI models that applies optimized subword tokenization for real-world LLM applications.


# 2: What is a Language Model?

A language model is designed to predict the next token based on the sequence of tokens it has already received. Given an input sequence `[t₁, t₂, …, tₙ]`, the model estimates the probability of the next token `tₙ₊₁`.

Models such as GPT-2 are built using multiple Transformer layers. These layers use attention mechanisms to capture relationships between tokens and feed-forward networks to refine the learned representations. The final output of the model is a set of **logits**, where each value represents the likelihood of a token being selected next.

In this section, I will load the GPT-2 model, examine its architecture, calculate the number of parameters, and observe how it predicts the next token for a given input.


### 2.1: Loading GPT-2

There are several ways to load and work with pretrained language models. For this project, I used the **Hugging Face Transformers** library because it provides a simple interface for downloading and running models such as GPT-2.

Besides Hugging Face, there are specialized inference engines like **Ollama**, **SGLang**, and **vLLM** that are designed to serve large language models more efficiently. I will explore these tools in later projects.

In the next step, I load the GPT-2 model and examine its architecture to understand how it is organized internally.


In [15]:
import torch
from transformers import GPT2LMHeadModel

# Step 1: Load the pretrained GPT-2 model.
# Reference: https://huggingface.co/docs/transformers/en/model_doc/gpt2

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

# Step 2: Display the complete GPT-2 model architecture.
print("=== GPT-2 Model Architecture ===")
print(gpt2, "\n")

# Step 3: Examine the first Transformer block and its components.
block = gpt2.transformer.h[0]  # GPT-2 consists of 12 Transformer blocks.

print("=== First Transformer Block ===")
print(block, "\n")

print("Components in the First Transformer Block:")
for name, module in block.named_children():
    print(f"{name:7s} -> {module.__class__.__name__}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

=== GPT-2 Model Architecture ===
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
) 

=== First Transfor

### 2.2: Counting Parameters

The size of a language model is commonly described by the number of parameters it contains. These parameters are the weights and biases that the model learns during training, allowing it to recognize patterns and generate meaningful text.

In this section, I calculate the total number of parameters in GPT-2 and examine how they are distributed across different components of the model. This provides a better understanding of where most of the model's capacity comes from.


In [16]:
# Count the total number of trainable parameters in the GPT-2 model.

total = sum(p.numel() for p in gpt2.parameters())

print(f"Total Parameters: {total:,}")

Total Parameters: 124,439,808


**Think about scale:** GPT-2 Small contains approximately 124 million parameters, while modern language models can have tens or even hundreds of billions of parameters. Assuming each parameter is stored as a 16-bit floating-point value (2 bytes), estimate the amount of memory required to load GPT-2 into RAM. Then, perform the same calculation for a model with 70 billion parameters and compare the difference.


### 2.3: From Text to Predictions

When an input sequence is passed through a language model, the output is a tensor of **logits** with the shape `(batch_size, seq_len, vocab_size)`. Each position in the sequence contains a score for every token in the model's vocabulary, indicating how likely that token is to be predicted next.

These logits are converted into probabilities using the **softmax** function, which transforms the scores into values that add up to 1. The token with the highest probability becomes the model's most likely prediction.

In the next step, I pass a sample sentence to GPT-2, inspect the shape of the output logits, and identify the top five token predictions for the final position in the sequence.


In [18]:
from transformers import AutoTokenizer

# Step 1: Load the GPT-2 tokenizer.

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [19]:
# Step 2: Tokenize a sample input sentence.

text = "Artificial intelligence is transforming software development."

input_ids = tokenizer(text, return_tensors="pt").input_ids  # Shape: (1, sequence_length)


In [20]:
# Step 3: Pass the tokenized input to GPT-2 and obtain the output logits.

with torch.no_grad():
    logits = gpt2(input_ids).logits  # Shape: (batch_size, sequence_length, vocabulary_size)

print("Logits Shape:", logits.shape)

Logits Shape: torch.Size([1, 8, 50257])


In [21]:
import torch.nn.functional as F

# Step 4: Predict the next token by converting logits into probabilities
# and selecting the five most likely candidates.

probs = F.softmax(logits[0, -1], dim=-1)  # Probability distribution for the next token.
topk = torch.topk(probs, 5)

print("\nTop 5 Predicted Tokens:")
for idx, p in zip(topk.indices.tolist(), topk.values.tolist()):
    print(f"{tokenizer.decode([idx]):>10s} : {p:.4f}")


Top 5 Predicted Tokens:
        It : 0.0947
         
 : 0.0914
       The : 0.0756
        In : 0.0490
        We : 0.0275


### 2.4: Key Takeaway

A language model is built from a collection of neural network layers that work together to predict the next token in a sequence. Although these models contain millions or billions of parameters, their core objective remains the same: estimating the most likely next token based on the input context.

By training on massive amounts of text, the model learns patterns, relationships, and context in language. This enables it to generate responses that are fluent, coherent, and relevant to the given input.


# 3: Text Generation (Decoding)

Once a language model predicts the probability of the next token, those predictions can be used to generate complete text. This process is known as **text generation** or **decoding**.

The text generation process follows these steps:

1. Pass the current token sequence to the model.
2. Obtain the probability distribution for the next token.
3. Select the next token using a decoding strategy.
4. Append the selected token to the input sequence and repeat the process.

Libraries such as Hugging Face Transformers provide a `generate()` method that performs this entire workflow, including token generation, stopping criteria, and other optimizations.

The quality and style of the generated text depend on the decoding strategy:

* **Greedy decoding:** Always selects the token with the highest probability. This produces consistent results but can sometimes lead to repetitive text.
* **Top-p (nucleus) sampling:** Randomly selects the next token from a group of high-probability candidates whose cumulative probability exceeds a chosen threshold. This usually produces more natural and varied responses.

In the following sections, I use the `generate()` method with GPT-2 to compare the outputs produced by greedy decoding and top-p sampling.


### 3.1: Greedy Decoding

In this section, I use GPT-2 and the Hugging Face `generate()` method to generate text using the greedy decoding strategy. Since greedy decoding always selects the most probable next token, the output is deterministic and remains the same for the same input prompt.


In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"

# Step 1: Load the GPT-2 model and tokenizer.

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).eval().to(device)

# Step 2: Tokenize the prompt and generate text using greedy decoding.
prompt = "Artificial intelligence"

input_tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
    **input_tokens,
    max_new_tokens=10,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("Prompt :", prompt)
print("Output :", output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt : Artificial intelligence
Output : Artificial intelligence is a new field of research that has been in


In [25]:
# Step 3: Create a helper function for greedy decoding and
# test it with different prompts to observe the generated output.

test_prompts = [
    "Artificial intelligence can",
    "The future of technology is",
    "Explain machine learning in simple terms."
]

def generate(model, tokenizer, prompt, max_new_tokens=32):
    # Generate text using greedy decoding.
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("Decoding Strategy: Greedy")
    print(generate(model, tokenizer, prompt, max_new_tokens=80))


Prompt: Artificial intelligence can
Decoding Strategy: Greedy
Artificial intelligence can be used to solve problems that are difficult to solve.

The problem is that artificial intelligence is not a new concept. It has been around for a long time. The idea is that we can solve problems that are difficult to solve.

The problem is that we can solve problems that are difficult to solve.

The problem is that we can solve problems that are difficult to solve.

Prompt: The future of technology is
Decoding Strategy: Greedy
The future of technology is not yet clear. But it is clear that the future of the Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet clear.

The Internet is not yet

Prompt: Explain machine learning in simple terms.
Decoding Strategy: Greedy
Explain machine learning in simple terms.

The first step is to understan

Although greedy decoding is simple and deterministic, it has some practical limitations:

* **Repetitive output:** The model may repeatedly generate the same words or phrases, reducing the quality of the response.
* **Limited exploration:** Since it always chooses the highest-probability token, it may overlook alternative tokens that could produce more natural or coherent text.

To overcome these limitations, the next section explores **top-p (nucleus) sampling**, a decoding strategy that introduces controlled randomness while maintaining meaningful and fluent text generation.


### 3.2: Top-p (Nucleus) Sampling

Top-p sampling, also known as **nucleus sampling**, generates text by introducing a controlled level of randomness. Instead of selecting only the highest-probability token, the model samples from a group of likely tokens whose combined probability reaches a specified threshold `p` (for example, 0.9).

This approach allows the model to produce more varied and natural responses while maintaining overall coherence.

In this section, I implement a text generation function that supports both **greedy decoding** and **top-p sampling**, making it easy to compare the behavior of the two decoding strategies.


In [26]:
# Generate text using top-p (nucleus) sampling.
# Reference: https://huggingface.co/docs/transformers/en/main_classes/text_generation

prompt = "Artificial intelligence"

input_tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
    **input_tokens,
    max_new_tokens=10,
    do_sample=True,
    top_p=0.9,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id
)

output = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("Prompt :", prompt)
print("Output :", output)

Prompt : Artificial intelligence
Output : Artificial intelligence (AI) is the fastest growing area of research


In [27]:
# Create a helper function for text generation using different decoding strategies.

def generate(model, tokenizer, prompt, strategy="top_p", max_new_tokens=128):
    # Tokenize the input prompt.
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen_args = dict(
        **enc,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    # Configure the decoding strategy.
    if strategy == "greedy":
        gen_args["do_sample"] = False
    elif strategy == "top_p":
        gen_args.update(
            dict(
                do_sample=True,
                top_p=0.9,
                temperature=0.9
            )
        )

    # Generate and decode the output.
    output = model.generate(**gen_args)
    return tokenizer.decode(output[0], skip_special_tokens=True)


test_prompts = [
    "Artificial intelligence can",
    "The future of software development is",
    "Explain cloud computing in simple terms."
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("Decoding Strategy: Top-p")
    print(generate(model, tokenizer, prompt, strategy="top_p", max_new_tokens=32))


Prompt: Artificial intelligence can
Decoding Strategy: Top-p
Artificial intelligence can be used to perform tasks in a variety of ways, including self-driving cars and autonomous vehicles. However, a number of technical barriers to using AI, such

Prompt: The future of software development is
Decoding Strategy: Top-p
The future of software development is becoming more and more difficult to predict. The first question we're asked at the beginning of every year is "How do we make this easier?" We can't

Prompt: Explain cloud computing in simple terms.
Decoding Strategy: Top-p
Explain cloud computing in simple terms. There are plenty of resources on the web to learn about cloud computing, but one important thing that is not well understood is the nature of the service.




# 4: From Simple GPT-2 to Modern LLMs

So far, I have been working with **GPT-2**, an early autoregressive language model released in 2019 with approximately 124 million parameters. GPT-2 is designed for text completion, meaning it predicts the next token based only on the input it receives. It does not distinguish between a question, an instruction, or a conversation, so it simply continues the text.

Modern language models, such as **Qwen3-0.6B** (600 million parameters), have been further trained to support a much wider range of tasks. In addition to text completion, they can:

* **Follow instructions** by interpreting user requests and producing relevant responses.
* **Maintain conversations** by using the context from previous interactions.
* **Perform reasoning** by breaking down complex problems before generating a final answer.

In this section, I compare GPT-2 and Qwen3-0.6B using the same prompts to observe how instruction tuning changes the model's behavior.

### 4.1: Chat Templates

Unlike GPT-2, instruction-tuned models expect input in a structured chat format instead of plain text. Rather than passing a prompt directly, the input is formatted as a conversation between the user and the assistant.

For example:

```text
<|user|>What is 2+2?<|assistant|>
```

Each model family defines its own chat template. The Hugging Face `tokenizer.apply_chat_template()` method automatically applies the correct format, allowing the model to interpret the input as a user request instead of ordinary text.

### 4.2: GPT-2 vs. Qwen3-0.6B

In the following examples, I use the same prompt with both models to compare their responses.

* **GPT-2:** Continues the input text without recognizing it as a question or instruction.
* **Qwen3-0.6B:** Receives a properly formatted chat prompt, interprets the user's request, and generates a direct response.


In [28]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the GPT-2 and Qwen models along with their tokenizers.
# Reference: https://huggingface.co/docs/transformers

MODELS = {
    "gpt2": "gpt2",
    "qwen": "Qwen/Qwen3-0.6B"
}

tokenizers = {}
models = {}

device = "cuda" if torch.cuda.is_available() else "cpu"

for key, model_id in MODELS.items():
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Use the end-of-sequence token as the padding token when required.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_id).eval().to(device)

    tokenizers[key] = tokenizer
    models[key] = model

    print(f"Loaded model: {model_id}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded model: gpt2


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-0.6B


Both models are now available for testing. GPT-2 contains approximately 124 million parameters, while Qwen3-0.6B has around 600 million parameters. If loading the models took some time, it was most likely because they were being downloaded for the first time. Once downloaded, they are stored in the local cache, making subsequent executions much faster.

Next, I generate text using the same prompt with both models. GPT-2 receives the prompt as plain text, whereas Qwen first formats the input using `apply_chat_template()` before generation. Both models use **top-p sampling**, making the generated responses more diverse and natural.


In [29]:
# Compare the responses generated by GPT-2 and Qwen using the same prompt.
# GPT-2 receives plain text, while Qwen uses a chat template.
# Reference: https://huggingface.co/docs/transformers/en/chat_templating

prompt = "A farmer has 17 sheep. All but 9 die. How many sheep are left?"

# ----- GPT-2: Generate a text completion from the raw prompt -----

gpt2_inputs = tokenizers["gpt2"](prompt, return_tensors="pt").to(device)

gpt2_output = models["gpt2"].generate(
    **gpt2_inputs,
    max_new_tokens=32,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizers["gpt2"].eos_token_id,
)

print("=== GPT-2 Output ===")
print(tokenizers["gpt2"].decode(gpt2_output[0], skip_special_tokens=True))


# ----- Qwen: Format the prompt as a chat conversation before generation -----

messages = [
    {"role": "user", "content": prompt}
]

chat_text = tokenizers["qwen"].apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

qwen_inputs = tokenizers["qwen"](chat_text, return_tensors="pt").to(device)

qwen_output = models["qwen"].generate(
    **qwen_inputs,
    max_new_tokens=1024,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizers["qwen"].eos_token_id,
)

qwen_new_tokens = qwen_output[0][qwen_inputs["input_ids"].shape[1]:]

print("\n=== Qwen3-0.6B Output ===")
print(tokenizers["qwen"].decode(qwen_new_tokens, skip_special_tokens=True))

=== GPT-2 Output ===
A farmer has 17 sheep. All but 9 die. How many sheep are left?

"This is a massive tragedy and a great tragedy," said the man, who has been identified as David D. Taylor, the owner of the farm

=== Qwen3-0.6B Output ===
<think>
Okay, let's see. The problem says a farmer has 17 sheep. All but 9 die. How many are left?

Hmm, so all but 9 die. That means if 9 die, then the number of sheep remaining would be 17 minus 9. Let me check that. If you have 17 and subtract 9, you get 8. So, 8 sheep are left. 

Wait, but maybe I should think again. Sometimes these problems can be tricky. Let me make sure. The question says "all but 9 die." So, "all" meaning all the sheep, except 9. So, total sheep minus the ones that died. So, 17 minus 9 equals 8. Yeah, that seems straightforward. 

Is there any catch here? Maybe the wording is confusing? Like, does "all but 9 die" mean that 9 are still alive? But that would be "all but 9" meaning 17 minus 9 is 8. So, no, that's not conflictin

# 5. (Optional) A Small Interactive LLM Playground

This section is optional and is intended for additional practice. Completing it is not required for understanding the core concepts covered in this project, but it provides an opportunity to experiment with language models in a more interactive way.

If you would like to extend the project, you can build a simple LLM playground with features such as:

* Input fields for entering prompts and selecting a model.
* Controls to choose a decoding strategy and adjust the temperature.
* Text generation using the Hugging Face `generate()` method.
* Displaying the generated response directly within the notebook.

**References:**

* https://ipywidgets.readthedocs.io/en/latest/
* https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html


In [30]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Build a simple interactive playground for experimenting with different
# language models and decoding strategies.

# Load models only if they are not already available.
try:
    tokenizers
    models
except NameError:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    MODELS = {
        "gpt2": "gpt2",
        "qwen": "Qwen/Qwen3-0.6B"
    }

    tokenizers = {}
    models = {}

    device = "cuda" if torch.cuda.is_available() else "cpu"

    for key, model_id in MODELS.items():
        tokenizer = AutoTokenizer.from_pretrained(model_id)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(model_id).eval().to(device)

        tokenizers[key] = tokenizer
        models[key] = model

        print(f"Loaded model: {model_id}")


# Generate text using the selected decoding strategy.
def generate_playground(model_key, prompt, strategy="greedy",
                        temperature=1.0, max_new_tokens=100):

    tokenizer = tokenizers[model_key]
    model = models[model_key]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    generation_args = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    if strategy == "greedy":
        generation_args["do_sample"] = False

    elif strategy == "top_k":
        generation_args.update(
            dict(
                do_sample=True,
                top_k=50,
                temperature=temperature
            )
        )

    elif strategy == "top_p":
        generation_args.update(
            dict(
                do_sample=True,
                top_p=0.9,
                temperature=temperature
            )
        )

    else:
        raise ValueError("Unsupported decoding strategy.")

    output = model.generate(**generation_args)

    return tokenizer.decode(output[0], skip_special_tokens=True)


# Create the interactive controls.

prompt_box = widgets.Textarea(
    value="Explain how transformers work.",
    placeholder="Enter your prompt...",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="120px")
)

model_dropdown = widgets.Dropdown(
    options=[
        ("GPT-2", "gpt2"),
        ("Qwen3-0.6B", "qwen")
    ],
    value="gpt2",
    description="Model:"
)

strategy_dropdown = widgets.Dropdown(
    options=[
        ("Greedy", "greedy"),
        ("Top-k", "top_k"),
        ("Top-p", "top_p")
    ],
    value="greedy",
    description="Strategy:"
)

temperature_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.1,
    description="Temperature:"
)

generate_button = widgets.Button(
    description="Generate",
    button_style="primary"
)

output_area = widgets.Output()


# Generate text when the button is clicked.
def on_generate(_):

    output_area.clear_output()

    with output_area:

        try:
            response = generate_playground(
                model_dropdown.value,
                prompt_box.value,
                strategy_dropdown.value,
                temperature_slider.value
            )

            display(Markdown(f"### Generated Response\n\n{response}"))

        except Exception as error:
            print(f"Error: {error}")


generate_button.on_click(on_generate)


# Arrange and display the playground.

ui = widgets.VBox([
    prompt_box,
    widgets.HBox([
        model_dropdown,
        strategy_dropdown,
        temperature_slider
    ]),
    generate_button,
    output_area
])

display(ui)

# 6: Inference Engines: Ollama, vLLM, SGLang

Throughout this project, I loaded language models directly in Python using the Hugging Face `transformers` library. This approach is ideal for understanding how models work, but real-world applications typically use dedicated inference servers instead of loading models inside every application.

An **inference engine** keeps the model loaded in memory, manages GPU resources, processes multiple requests efficiently, and exposes an HTTP API that client applications can use for text generation.

Some widely used inference engines include:

| Engine     | Primary Use                                                    |
| ---------- | -------------------------------------------------------------- |
| **Ollama** | Running and experimenting with LLMs locally                    |
| **vLLM**   | High-performance inference for production workloads            |
| **SGLang** | Efficient model serving with support for structured generation |

A key advantage of many inference engines is their support for the **OpenAI-compatible API**. This allows the same client code to communicate with different backends. For example, you can use Ollama during development and later switch to vLLM or SGLang for production without making significant changes to your application.

In future projects, I will explore Ollama in more detail, learn how to configure it, and use it to run and build applications with modern large language models.


## Wrapping Up !!

In this project, I explored the core concepts behind large language models and built a solid understanding of how text is processed and generated. The key concepts covered include:

* Understanding different **tokenization** techniques, from word-level tokenization to Byte-Pair Encoding (BPE).
* Comparing tokenizers used by different GPT model families with `tiktoken`.
* Loading and exploring the architecture of the GPT-2 model.
* Calculating the number of model **parameters** and understanding their significance.
* Examining how GPT-2 produces **logits** and converts them into probabilities for next-token prediction.
* Implementing and comparing **greedy decoding** and **top-p (nucleus) sampling**.
* Comparing GPT-2 with Qwen3-0.6B to observe the differences between a text completion model and an instruction-tuned language model.
* Learning the purpose of modern **inference engines** such as Ollama, vLLM, and SGLang.

Completing this project has given me a practical understanding of how large language models process input, predict tokens, and generate coherent text. These concepts provide a strong foundation for exploring more advanced topics and building LLM-powered applications in future projects.
